In [1]:
import os
from pathlib import Path

PROJECT_DIR = Path.cwd()
LOCAL_MODEL_DIR = PROJECT_DIR / "models" / "codebert_java_cls_ft"
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

HF_HOME = PROJECT_DIR / ".hf_home"
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

In [2]:

MODEL_NAME = "khanhtran0111/codebert-java-vul4j-ft" 

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

tokenizer.save_pretrained(LOCAL_MODEL_DIR)
model.save_pretrained(LOCAL_MODEL_DIR)

print("Saved to:", LOCAL_MODEL_DIR)
print("num_labels:", getattr(config, "num_labels", None))
print("id2label:", getattr(config, "id2label", None))
print("label2id:", getattr(config, "label2id", None))

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

c:\Users\Admin\Documents\miniconda\envs\nckh\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\LM\.hf_home\hub\models--khanhtran0111--codebert-java-vul4j-ft. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: c:\Users\Admin\Documents\1. UET\lab\VulGuardVN\LM\models\codebert_java_cls_ft
num_labels: 2
id2label: {0: 'LABEL_0', 1: 'LABEL_1'}
label2id: {'LABEL_0': 0, 'LABEL_1': 1}


In [3]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(LOCAL_MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

def predict(code: str, max_length: int = 512):
    inputs = tokenizer(code, truncation=True, max_length=max_length,
                       padding="max_length", return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(0)
    probs = F.softmax(logits, dim=-1).cpu().tolist()

    id2label = getattr(model.config, "id2label", None)
    if isinstance(id2label, dict) and len(id2label) == len(probs):
        pred_id = int(torch.argmax(logits).item())
        return id2label[pred_id], {id2label[i]: probs[i] for i in range(len(probs))}
    else:
        pred_id = int(torch.argmax(logits).item())
        return f"LABEL_{pred_id}", {f"LABEL_{i}": probs[i] for i in range(len(probs))}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
sample_java = r"""
import java.sql.*;
public class Demo {
  public static void main(String[] args) throws Exception {
    String user = args[0];
    String q = "SELECT * FROM users WHERE name = '" + user + "'";
    System.out.println(q);
  }
}
"""

pred_label, prob_map = predict(sample_java)
print("Prediction:", pred_label)
for k, v in sorted(prob_map.items(), key=lambda kv: -kv[1]):
    print(f"  {k}: {v:.4f}")

Prediction: LABEL_0
  LABEL_0: 0.9261
  LABEL_1: 0.0739


In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

code_box = widgets.Textarea(
    value="",
    placeholder="Dán Java code nhiều dòng vào đây...",
    description="Java:",
    layout=widgets.Layout(width="100%", height="260px"),
)

run_btn = widgets.Button(description="Run", button_style="primary")
clear_btn = widgets.Button(description="Clear")
out = widgets.Output()

def on_run(_):
    with out:
        clear_output()
        code = code_box.value.strip()
        if not code:
            print("(Empty input) — thử lại.")
            return
        pred_label, prob_map = predict(code)
        print("=== RESULT ===")
        print("Prediction:", pred_label)
        for k, v in sorted(prob_map.items(), key=lambda kv: -kv[1]):
            print(f"  {k}: {v:.4f}")
        print("=============")

def on_clear(_):
    code_box.value = ""
    with out:
        clear_output()

run_btn.on_click(on_run)
clear_btn.on_click(on_clear)

display(widgets.VBox([code_box, widgets.HBox([run_btn, clear_btn]), out]))


In [4]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
